# Lab: Build a Voice Assistant with Whisper

### `Speech In -> Think -> Speech Out` | One-Day Guided Lab

By the end of today you will have a working voice assistant that you can **talk to**
and that **talks back**. Three stages, three models:

```
                     +--------------+     +--------------+     +--------------+
  your voice  ---->  |   WHISPER    | --> |     LLM      | --> |     TTS      |  ----> speaker
   (16 kHz wav)      |  speech->text|     |   the brain  |     |  text->speech|
                     +--------------+     +--------------+     +--------------+
                          "ears"              "thinking"            "mouth"
```

Each stage is ~15 lines of code. The interesting work is everything *around* them:
handling silence, keeping conversation memory, shaping text so it *sounds* right when
spoken, and keeping total latency low enough that the conversation feels alive.

---

**Level:** beginner to intermediate. You need Python and basic NumPy. No prior audio or
speech-processing experience required.

**Time:** ~6 hours including breaks.

**Runs on:** your laptop (CPU is fine) or Google Colab. A GPU makes Whisper faster but is
not required for the small models we use.

## Today's schedule

| Part | What you build | Time |
|---|---|---|
| 0 | Setup and environment check | 20 min |
| 1 | Getting audio in (mic, file, or synthesised) | 30 min |
| 2 | **Whisper**: the ears | 75 min |
| 3 | The brain: an LLM with conversation memory | 60 min |
| 4 | The mouth: text-to-speech | 40 min |
| 5 | Assembling the loop + measuring latency | 60 min |
| 6 | Making it feel real: endpointing and push-to-talk | 45 min |
| 7 | Stretch challenges and submission | 30 min |

## Learning outcomes

After this lab you should be able to:

1. Transcribe audio with Whisper and read its segment-level output, including timestamps
   and the `no_speech_prob` confidence signal.
2. Explain, in your own words, what a log-Mel spectrogram is and why Whisper uses one.
3. Choose a Whisper model size using a measured speed/accuracy trade-off rather than a guess.
4. Write a system prompt tuned for **spoken** output rather than written output.
5. Maintain multi-turn conversation memory within a token budget.
6. Profile a three-stage pipeline and identify which stage owns your latency.

## How to work through this

Cells marked **`# TODO`** are yours to complete. Everything else is given to you and
should run as-is. If you get stuck for more than 10 minutes on a TODO, ask -- the point
is to finish the pipeline, not to be blocked on one function.

---
# Part 0 - Setup (20 min)

We deliberately use packages that work **offline after the first download**, so the lab
does not fall over if the classroom wifi does.

Run the install cell once. It takes 2-5 minutes.

In [1]:
# Run once. On Colab, drop the leading '#' from the ffmpeg line too.
%pip install -q openai-whisper sounddevice scipy matplotlib numpy requests gTTS

# ffmpeg is a *system* package that Whisper shells out to for decoding audio.
#   Colab / Ubuntu :  !apt-get -qq install -y ffmpeg
#   macOS          :  brew install ffmpeg
#   Windows        :  winget install ffmpeg     (then restart the kernel)

# Optional extras:
#   pyttsx3   -> fully offline text-to-speech (no internet needed at run time)
#   anthropic -> if you are using the Claude API instead of a local model
# %pip install -q pyttsx3 anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Environment check

This next cell does not install anything. It just reports **what you actually have**, so
you know which paths through the lab are open to you. Nothing here should raise -- if a
component is missing you will see `no` next to it and we will route around it.

In [ ]:
import importlib, shutil, subprocess, sys

def _has_module(name):
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False

def _has_ollama():
    """Is a local Ollama server answering on the default port?"""
    try:
        import requests
        requests.get("http://localhost:11434/api/tags", timeout=2).raise_for_status()
        return True
    except Exception:
        return False

CAPS = {
    "python >= 3.9":  sys.version_info >= (3, 9),
    "whisper":        _has_module("whisper"),
    "ffmpeg (system)": shutil.which("ffmpeg") is not None,
    "numpy":          _has_module("numpy"),
    "matplotlib":     _has_module("matplotlib"),
    "torch":          _has_module("torch"),
    "sounddevice (mic)": _has_module("sounddevice"),
    "gTTS (online TTS)": _has_module("gtts"),
    "pyttsx3 (offline TTS)": _has_module("pyttsx3"),
    "Ollama server":  _has_ollama(),
    "anthropic SDK":  _has_module("anthropic"),
}

width = max(len(k) for k in CAPS)
print("ENVIRONMENT REPORT")
print("-" * (width + 8))
for name, ok in CAPS.items():
    print(f"{name:<{width}}  {'yes' if ok else 'no '}")
print("-" * (width + 8))

if not CAPS["whisper"] or not CAPS["ffmpeg (system)"]:
    print("\n! Whisper needs BOTH the python package and the ffmpeg binary. See the cell above.")
if not (CAPS["Ollama server"] or CAPS["anthropic SDK"]):
    print("\n  No LLM backend detected -> Part 3 will use the built-in scripted fallback.")
    print("  That is fine for today; the pipeline still runs end to end.")
if not (CAPS["gTTS (online TTS)"] or CAPS["pyttsx3 (offline TTS)"]):
    print("\n  No TTS engine -> replies will be printed instead of spoken.")

> **Note on the fallbacks.** This notebook is written so that *every* stage degrades
> gracefully: no mic, no LLM, and no TTS engine all have working substitutes. You will
> still get a complete pipeline. Do not skip a part just because one component is missing.

---
# Part 1 - Getting audio in (30 min)

Whisper wants **16 kHz mono** audio. That is the first thing to internalise: 16,000
amplitude samples per second, one channel. Music is usually 44.1 kHz stereo; speech
carries almost no useful information above 8 kHz, so 16 kHz is plenty and it keeps the
model small.

You have three ways to get a clip. Use whichever works on your machine:

- **A. Synthesise one** - works everywhere, no mic, no download. Start here.
- **B. Record from your mic** - the real thing.
- **C. Bring your own `.wav`** - drop a file next to the notebook.

In [ ]:
import os
import numpy as np

SAMPLE_RATE = 16_000          # Whisper's native rate. Everything downstream assumes this.
AUDIO_DIR = "audio"
os.makedirs(AUDIO_DIR, exist_ok=True)


# ---------- A. synthesise a test clip (no mic needed) ------------------------
def make_test_clip(text="What is the capital of France, and why is it famous?",
                   path=f"{AUDIO_DIR}/test_input.wav"):
    """Speak `text` to a file so we have something to transcribe.

    Uses gTTS (needs internet) and falls back to pyttsx3 (fully offline).
    Returns the path, or None if no engine is available.
    """
    try:
        from gtts import gTTS
        mp3 = path.replace(".wav", ".mp3")
        gTTS(text=text, lang="en").save(mp3)
        # Whisper reads mp3 fine via ffmpeg, so we can just hand back the mp3.
        print(f"[gTTS] wrote {mp3}")
        return mp3
    except Exception as e:
        print(f"[gTTS] unavailable ({type(e).__name__})")

    try:
        import pyttsx3
        eng = pyttsx3.init()
        eng.save_to_file(text, path)
        eng.runAndWait()
        print(f"[pyttsx3] wrote {path}")
        return path
    except Exception as e:
        print(f"[pyttsx3] unavailable ({type(e).__name__})")

    print("No TTS engine -> record with your mic (option B) or supply a .wav (option C).")
    return None


clip_path = make_test_clip()
clip_path

In [ ]:
# ---------- B. record from your microphone -----------------------------------
def record(seconds=5, sr=SAMPLE_RATE, path=f"{AUDIO_DIR}/mic_input.wav"):
    """Block for `seconds`, capture mono audio, write a 16-bit PCM wav."""
    import sounddevice as sd
    from scipy.io import wavfile

    print(f"Recording {seconds}s -- speak now...")
    audio = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    print("done.")

    audio = audio.flatten()
    # float32 in [-1, 1]  ->  int16 in [-32768, 32767], the standard wav encoding
    wavfile.write(path, sr, (audio * 32767).astype(np.int16))
    return path


# Uncomment to record. (This will not work on Colab -- Colab has no direct mic access.)
# clip_path = record(seconds=5)

In [ ]:
# ---------- C. use your own file ---------------------------------------------
# clip_path = "audio/my_question.wav"

assert clip_path and os.path.exists(clip_path), (
    "No audio clip yet. Run option A, B, or C above before continuing."
)
print("Using:", clip_path, f"({os.path.getsize(clip_path)/1024:.1f} KB)")

### Look at the sound before you model it

A good habit with any new data type: **plot it first**. Audio is just a 1-D array of
numbers between -1 and 1. Speech looks like bursts of energy (words) separated by
near-flat stretches (pauses) -- and those pauses are going to matter to us in Part 6.

In [ ]:
import matplotlib.pyplot as plt
import whisper

# whisper.load_audio handles ANY format ffmpeg can read and resamples to 16 kHz mono.
audio = whisper.load_audio(clip_path)          # -> float32 numpy array, values in [-1, 1]
duration = len(audio) / SAMPLE_RATE

print(f"shape    : {audio.shape}")
print(f"dtype    : {audio.dtype}")
print(f"duration : {duration:.2f} s")
print(f"range    : [{audio.min():.3f}, {audio.max():.3f}]")

t = np.arange(len(audio)) / SAMPLE_RATE
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.5)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude")
ax.set_title(f"Waveform - {os.path.basename(clip_path)}")
ax.margins(x=0)
plt.tight_layout()
plt.show()

**Look at the plot and answer for yourself:** can you point to where each word is? Where
are the pauses? Roughly how loud is the background noise between words compared to the
speech itself? That ratio is the entire basis of the silence detector you will write in
Part 6.

---
# Part 2 - Whisper: the ears (75 min)

Whisper is an open-source speech recognition model released by OpenAI in 2022. Two things
make it unusually good for a project like this:

1. **It was trained on 680,000 hours of audio** scraped from the web, in 98 languages.
   That is roughly 100x more data than the academic benchmarks of the time, and most of it
   is messy real-world audio -- accents, background noise, bad microphones. So it is
   robust out of the box, with no fine-tuning.
2. **It is multitask.** The same weights do transcription, translation-into-English,
   language identification, and timestamp prediction. Which task you get is controlled by
   *special tokens fed to the decoder*, not by a different model.

### Model sizes

| Model | Parameters | Rough VRAM | Relative speed | Multilingual? |
|---|---|---|---|---|
| `tiny` | 39 M | ~1 GB | ~10x | yes (`tiny.en` = English only) |
| `base` | 74 M | ~1 GB | ~7x | yes |
| `small` | 244 M | ~2 GB | ~4x | yes |
| `medium` | 769 M | ~5 GB | ~2x | yes |
| `large-v3` | 1550 M | ~10 GB | 1x | yes |
| `turbo` | 809 M | ~6 GB | ~8x | yes (transcription only, no translation) |

The `.en` variants (`tiny.en`, `base.en`, `small.en`, `medium.en`) are English-only and
noticeably better than their multilingual counterparts *at English*, because they do not
spend capacity on 97 other languages.

**For a live voice assistant, start at `base` or `base.en`.** Speed matters more than the
last few percent of accuracy when someone is waiting for a reply.

In [ ]:
import time
import whisper

MODEL_NAME = "base"      # try "tiny" if loading is slow, "small" if you have time

t0 = time.time()
model = whisper.load_model(MODEL_NAME)     # downloads weights on first run (~140 MB for base)
load_s = time.time() - t0

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded '{MODEL_NAME}' in {load_s:.1f}s")
print(f"parameters : {n_params/1e6:.1f} M")
print(f"device     : {next(model.parameters()).device}")
print(f"multilingual: {model.is_multilingual}")

In [ ]:
t0 = time.time()
result = model.transcribe(clip_path)
infer_s = time.time() - t0

print(f"transcription took {infer_s:.2f}s for {duration:.2f}s of audio")
print(f"real-time factor  : {infer_s/duration:.2f}x   (lower is better; <1.0 = faster than real time)")
print()
print("TEXT:", result["text"].strip())

### What actually came back

`transcribe()` returns a dictionary. The three keys you care about:

- **`text`** - the whole transcript as one string. This is what you will feed the LLM.
- **`language`** - the detected language code, e.g. `"en"`.
- **`segments`** - a list of chunks, each with its own start/end timestamps and a set of
  diagnostic scores. This is where the useful signal lives.

The per-segment field to know is **`no_speech_prob`**: the model's own estimate that this
segment contains *no speech at all*. Whisper is famous for confidently transcribing
silence into plausible-sounding sentences (often the training-data artefacts it saw most,
like subtitle credits). Filtering on `no_speech_prob` is the standard defence, and you
will implement it in a moment.

In [ ]:
print(f"detected language : {result['language']}")
print(f"segments          : {len(result['segments'])}\n")

hdr = f"{'#':>2}  {'start':>7}  {'end':>7}  {'no_speech':>9}  {'avg_logprob':>11}  text"
print(hdr)
print("-" * (len(hdr) + 20))
for seg in result["segments"]:
    print(f"{seg['id']:>2}  {seg['start']:>7.2f}  {seg['end']:>7.2f}  "
          f"{seg['no_speech_prob']:>9.3f}  {seg['avg_logprob']:>11.3f}  {seg['text'].strip()}")

### How Whisper works (15 min - read this properly)

Whisper is a plain **encoder-decoder Transformer**, the same architecture as the original
"Attention Is All You Need" machine-translation model. The clever part is not the
architecture; it is how audio and tasks are represented.

**Step 1 - audio becomes a picture.** The waveform is padded or trimmed to exactly **30
seconds**, then converted to a **log-Mel spectrogram**: a 2-D array of
`80 mel bins x 3000 time frames` (`large-v3` uses 128 bins). Read that as an image of
sound -- x-axis time, y-axis pitch, brightness = energy.

Why Mel and why log?

- The **Mel scale** spaces the frequency bins the way human hearing does: fine resolution
  down where vowels and formants live, coarse resolution up high. It throws away detail we
  cannot hear anyway.
- The **log** compresses the enormous dynamic range of loudness into something a neural
  net can learn from, again matching human perception -- we hear loudness roughly
  logarithmically.

**Step 2 - the encoder reads the picture.** Two convolution layers downsample it, then a
stack of Transformer blocks with self-attention produces one vector per ~20 ms of audio.
Every position can attend to every other, so the encoder has the *whole* 30-second window
in view at once. That global context is why Whisper handles accents and noise well: it can
use later audio to disambiguate earlier audio.

**Step 3 - the decoder writes text**, one token at a time, cross-attending to the encoder
output. And here is the multitask trick -- the decoder is *primed* with special tokens:

```
<|startoftranscript|> <|en|> <|transcribe|> <|notimestamps|>  The quick brown fox ...
                       ^^^^   ^^^^^^^^^^^^
                    language     task
```

Swap `<|transcribe|>` for `<|translate|>` and the same weights emit English text for
Spanish audio. Leave the language token off and the model *predicts* it -- that is how
language detection works. Drop `<|notimestamps|>` and it emits timestamp tokens
interleaved with the words. **One model, four behaviours, selected by prompt.**

In [ ]:
# Let's actually look at what the encoder sees.
mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(audio))

print(f"mel shape: {tuple(mel.shape)}   -> (mel_bins, time_frames)")
print(f"3000 frames over 30 s = one frame per {30/mel.shape[1]*1000:.0f} ms")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)

axes[0].plot(np.arange(len(audio)) / SAMPLE_RATE, audio, linewidth=0.5)
axes[0].set_title("Waveform (what the microphone recorded)")
axes[0].set_ylabel("amplitude")
axes[0].margins(x=0)

im = axes[1].imshow(mel.numpy(), aspect="auto", origin="lower", cmap="magma",
                    extent=[0, 30, 0, mel.shape[0]])
axes[1].set_title("Log-Mel spectrogram (what Whisper's encoder reads)")
axes[1].set_xlabel("time (s)  -- note it is padded to the full 30 s window")
axes[1].set_ylabel("mel bin")
fig.colorbar(im, ax=axes[1], label="log energy")

plt.tight_layout()
plt.show()

**Read the spectrogram.** The horizontal stripes stacked at regular intervals during
vowels are *harmonics* -- the vibration of the vocal folds and its multiples. The wide
smeared blobs are fricatives (`s`, `sh`, `f`), which are basically shaped noise. The flat
dark region on the right is the zero-padding out to 30 seconds.

Notice that a 2-second clip and a 25-second clip cost Whisper **exactly the same compute**,
because both get padded to 30 seconds. That is a real design constraint for a voice
assistant: short utterances are inefficient. (Libraries like `faster-whisper` optimise
around this.)

### Exercise 2.1 - measure the size/speed trade-off (15 min)

Do not take the table above on faith. Measure it.

In [ ]:
# TODO: complete the loop below.
#   For each model size: load it, transcribe `clip_path`, and record
#   (a) load time, (b) inference time, (c) the transcript text.
#   Then print a comparison table.
#
# Watch out: the FIRST run of each size also downloads the weights, so time
# the load and the inference separately or your numbers will be nonsense.

SIZES = ["tiny", "base"]        # add "small" if you have the patience / a GPU
bench = []

for size in SIZES:
    t0 = time.time()
    m = whisper.load_model(size)
    t_load = time.time() - t0

    t0 = time.time()
    r = m.transcribe(clip_path)
    t_infer = time.time() - t0

    bench.append({
        "size": size,
        "params_M": sum(p.numel() for p in m.parameters()) / 1e6,
        "load_s": t_load,
        "infer_s": t_infer,
        "rtf": t_infer / duration,
        "text": r["text"].strip(),
    })
    del m                        # free memory before loading the next one

print(f"{'size':<8}{'params(M)':>10}{'load(s)':>9}{'infer(s)':>10}{'RTF':>7}")
for b in bench:
    print(f"{b['size']:<8}{b['params_M']:>10.1f}{b['load_s']:>9.1f}"
          f"{b['infer_s']:>10.2f}{b['rtf']:>7.2f}")

print("\nTranscripts:")
for b in bench:
    print(f"  [{b['size']:<5}] {b['text']}")

**Write down your answer:** at what point does the extra accuracy stop being worth the
extra latency, for a *conversational* assistant where the user is waiting? There is no
single right answer -- justify yours with your measured numbers.

### The other two tasks

Same weights, different decoder priming.

In [ ]:
# --- language identification (no transcription needed) -----------------------
mel30 = whisper.log_mel_spectrogram(whisper.pad_or_trim(audio)).to(model.device)
_, probs = model.detect_language(mel30)

top5 = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:5]
print("Top language guesses:")
for lang, p in top5:
    print(f"  {lang:<6} {p:.4f}")

# --- translate-into-English --------------------------------------------------
# On English audio this is a no-op. Try it on a clip in another language and the
# SAME model weights will emit English.
translated = model.transcribe(clip_path, task="translate")
print("\ntask='translate' ->", translated["text"].strip())

### Exercise 2.2 - defend against hallucinated silence (20 min)

Try this: record 5 seconds of **pure silence** (or pass an array of zeros) and transcribe
it. Whisper will often return text anyway -- frequently something like a subtitle credit,
because that is what accompanied silent stretches in its training data.

For a voice assistant this is a real bug: the user says nothing, and the assistant
confidently answers a question that was never asked. Write a wrapper that refuses to
return a transcript it does not believe in.

In [ ]:
# See the failure mode for yourself.
silence = np.zeros(SAMPLE_RATE * 5, dtype=np.float32)
noisy_silence = (np.random.randn(SAMPLE_RATE * 5) * 0.001).astype(np.float32)

for name, sig in [("pure silence", silence), ("quiet noise", noisy_silence)]:
    r = model.transcribe(sig, fp16=False)
    nsp = [round(s["no_speech_prob"], 3) for s in r["segments"]]
    print(f"{name:<14} text={r['text'].strip()!r:<45} no_speech_prob={nsp}")

In [ ]:
def robust_transcribe(audio_or_path, model, no_speech_threshold=0.6,
                      min_logprob=-1.0, min_chars=2):
    """Transcribe, but return '' when the model does not actually hear speech.

    Rejection rules:
      1. drop any segment whose no_speech_prob exceeds `no_speech_threshold`
      2. drop any segment whose avg_logprob is below `min_logprob` (gibberish)
      3. return '' if what survives is shorter than `min_chars`
    """
    result = model.transcribe(audio_or_path, fp16=False)

    # TODO: implement rules 1 and 2. Keep only segments that pass BOTH,
    #       then join their 'text' fields.
    kept = [
        seg for seg in result["segments"]
        if seg["no_speech_prob"] <= no_speech_threshold
        and seg["avg_logprob"] >= min_logprob
    ]

    text = " ".join(seg["text"].strip() for seg in kept).strip()
    return text if len(text) >= min_chars else ""


# Should print an empty string for both silence cases, and real text for your clip.
print(f"silence     -> {robust_transcribe(silence, model)!r}")
print(f"quiet noise -> {robust_transcribe(noisy_silence, model)!r}")
print(f"your clip   -> {robust_transcribe(clip_path, model)!r}")

> **Tune the threshold yourself.** `0.6` is a starting point, not a law. Too low and the
> assistant ignores quiet speakers; too high and it answers the air conditioning. Record a
> few clips at different volumes and find the value that works in *your* room.

---
# Part 3 - The brain (60 min)

Whisper gives us a string. Now something has to decide what to say back.

We support three backends, tried in order. **You only need one.**

1. **Ollama** (recommended) - runs a small model entirely on your machine. Free, private,
   no API key, no rate limits. Install from `ollama.com`, then `ollama pull llama3.2`.
2. **Claude API** - better quality, needs an API key and internet.
3. **Scripted fallback** - a tiny rule-based responder built into this notebook so the lab
   still works with no LLM at all.

In [ ]:
import os, requests

OLLAMA_URL   = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2"          # or "llama3.1", "phi3", "qwen2.5" -- whatever you pulled


def chat_ollama(messages, system, max_tokens=200):
    r = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "system", "content": system}] + messages,
            "stream": False,
            "options": {"num_predict": max_tokens, "temperature": 0.7},
        },
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["message"]["content"].strip()


def chat_anthropic(messages, system, max_tokens=200):
    from anthropic import Anthropic
    client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",   # small+fast; latency matters for voice.
        max_tokens=max_tokens,               # Check docs.claude.com for current model names.
        system=system,
        messages=messages,
    )
    return "".join(b.text for b in resp.content if b.type == "text").strip()


def chat_fallback(messages, system, max_tokens=200):
    """No LLM available. Keyword matching, so the pipeline still completes."""
    user = messages[-1]["content"].lower()
    table = [
        (("hello", "hi ", "hey"),        "Hello. I am a very small assistant. What can I do for you?"),
        (("time",),                      "I cannot see a clock, but your computer can."),
        (("weather",),                   "I have no window and no internet, so your guess beats mine."),
        (("capital", "france", "paris"), "Paris is the capital of France."),
        (("your name", "who are you"),   "I am a lab assistant built from Whisper, a language model, and a speech synthesiser."),
        (("thank",),                     "You are very welcome."),
        (("bye", "goodbye"),             "Goodbye. Nice talking with you."),
    ]
    for keys, reply in table:
        if any(k in user for k in keys):
            return reply
    return "I heard you say: " + messages[-1]["content"] + ". I do not have a language model loaded to reason about that."


def chat(messages, system, max_tokens=200):
    """Try each backend in order; return (reply_text, backend_name)."""
    for fn, name in ((chat_ollama, "ollama"),
                     (chat_anthropic, "anthropic"),
                     (chat_fallback, "fallback")):
        try:
            return fn(messages, system, max_tokens), name
        except Exception as e:
            print(f"  [{name} unavailable: {type(e).__name__}]")
    raise RuntimeError("unreachable -- fallback never fails")

### The most important idea in Part 3: **write for the ear, not the eye**

Almost every LLM you have used is tuned to produce *written* output -- headers, bullet
lists, bold text, code blocks, three well-organised paragraphs. Feed any of that to a
speech synthesiser and it is unbearable. The TTS engine will read asterisks as pauses,
plough through a 200-word answer with no way to skim, and lose the listener in the first
ten seconds.

Spoken language is different from written language in specific, fixable ways:

| Written answer | Spoken answer |
|---|---|
| Long; the reader can skim | Short; the listener cannot skim |
| Markdown structure carries meaning | Structure must be carried by words ("first... then...") |
| Numerals and symbols are fine (`$4.5M`, `~20%`) | Must be spelled out ("four and a half million dollars") |
| Lists are efficient | Lists are exhausting past about three items |
| Reader can re-read a hard sentence | Listener gets one pass -- short sentences only |

So the system prompt does most of the work here.

In [ ]:
SYSTEM_PROMPT = """You are a friendly voice assistant. Your replies are converted to \
speech and spoken aloud, so you must write the way a person talks, not the way a person writes.

Rules:
- Answer in at most two short sentences unless the user explicitly asks for detail.
- Never use markdown, bullet points, numbered lists, headers, emoji, or code blocks.
- Write out numbers, symbols and abbreviations as words: say "twenty percent", not "20%".
- Use contractions and plain everyday vocabulary.
- If you do not know something, say so in one sentence. Do not speculate at length.
- Never mention that you are an AI model unless you are asked directly."""

print(SYSTEM_PROMPT)

### Conversation memory

An LLM has no memory of its own. "Memory" is just: resend the whole conversation each
turn. That grows without bound, so we trim -- keeping the most recent turns, because in
spoken conversation the recent context is what pronouns refer to.

In [ ]:
class Memory:
    """Rolling conversation history, capped at `max_turns` user+assistant pairs."""

    def __init__(self, max_turns=6):
        self.max_turns = max_turns
        self.messages = []

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})
        # keep the last max_turns pairs = 2 * max_turns messages
        excess = len(self.messages) - 2 * self.max_turns
        if excess > 0:
            self.messages = self.messages[excess:]
            # a history must never start with an assistant turn
            while self.messages and self.messages[0]["role"] != "user":
                self.messages.pop(0)

    def clear(self):
        self.messages = []

    def __len__(self):
        return len(self.messages)

    def __repr__(self):
        return f"<Memory {len(self.messages)} messages>"


# quick sanity check
m = Memory(max_turns=2)
for i in range(5):
    m.add("user", f"question {i}")
    m.add("assistant", f"answer {i}")
print(m)
for msg in m.messages:
    print(f"  {msg['role']:<10} {msg['content']}")

In [ ]:
# Wire memory + chat together and try it.
memory = Memory(max_turns=6)

transcript = robust_transcribe(clip_path, model) or "Hello, who are you?"
print("USER:", transcript)

memory.add("user", transcript)
reply, backend = chat(memory.messages, SYSTEM_PROMPT)
memory.add("assistant", reply)

print(f"ASSISTANT [{backend}]:", reply)

### Exercise 3.1 - test the pronoun (10 min)

Multi-turn memory is easy to *think* you have implemented and easy to actually get wrong.
The classic test is a follow-up question that only makes sense with context.

In [ ]:
# TODO: run a two-turn conversation where the second turn contains a pronoun
#       that can only be resolved from the first turn.
#       e.g.  "Who wrote Pride and Prejudice?"  ->  "When was SHE born?"
#       Then set memory.clear() and ask ONLY the second question. Compare.

memory.clear()

for turn in ["Who wrote Pride and Prejudice?", "When was she born?"]:
    memory.add("user", turn)
    reply, _ = chat(memory.messages, SYSTEM_PROMPT)
    memory.add("assistant", reply)
    print(f"USER      : {turn}")
    print(f"ASSISTANT : {reply}\n")

### Exercise 3.2 - give it a personality (10 min)

Change `SYSTEM_PROMPT` so your assistant has a distinct character -- a terse ship's
computer, an over-enthusiastic tour guide, a sceptical librarian. **Keep every one of the
voice rules** (short, no markdown, numbers as words); only the persona changes. Then ask
it the same three questions and note how much the persona survives the two-sentence limit.

---
# Part 4 - The mouth (40 min)

Two engines, same trade-off as always:

- **gTTS** - Google's web endpoint. Very natural, needs internet, adds ~0.5-1.5 s of
  network latency per reply, and returns mp3.
- **pyttsx3** - drives your operating system's built-in synthesiser (SAPI5 on Windows,
  NSSpeechSynthesizer on macOS, espeak on Linux). Robotic, but completely offline and
  nearly instant.

For a *responsive* assistant, offline wins. For a *demo*, gTTS sounds better. We implement
both and let you switch.

In [ ]:
from IPython.display import Audio, display

TTS_ENGINE = "auto"      # "auto" | "gtts" | "pyttsx3" | "print"


def speak(text, path=f"{AUDIO_DIR}/reply.mp3", engine=None, autoplay=True):
    """Synthesise `text`. Returns the output path, or None if we could only print."""
    engine = engine or TTS_ENGINE

    if engine in ("auto", "gtts"):
        try:
            from gtts import gTTS
            gTTS(text=text, lang="en").save(path)
            if autoplay:
                display(Audio(path, autoplay=True))
            return path
        except Exception as e:
            if engine == "gtts":
                raise
            print(f"  [gTTS unavailable: {type(e).__name__}]")

    if engine in ("auto", "pyttsx3"):
        try:
            import pyttsx3
            wav = path.replace(".mp3", ".wav")
            eng = pyttsx3.init()
            eng.setProperty("rate", 175)      # words per minute; 175-190 sounds natural
            eng.setProperty("volume", 0.9)
            eng.save_to_file(text, wav)
            eng.runAndWait()
            if autoplay:
                display(Audio(wav, autoplay=True))
            return wav
        except Exception as e:
            if engine == "pyttsx3":
                raise
            print(f"  [pyttsx3 unavailable: {type(e).__name__}]")

    print(f"ASSISTANT (text only): {text}")
    return None


speak("Hello. I am your voice assistant, and this is what I sound like.")

### Exercise 4.1 - clean the text before you speak it (15 min)

Even with a good system prompt, models leak formatting. Write a sanitiser -- this is
unglamorous string work, and it is the difference between a demo that sounds finished and
one that sounds broken.

In [ ]:
import re

def clean_for_speech(text):
    """Strip formatting that a TTS engine would read aloud or stumble over."""
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)   # code blocks
    text = re.sub(r"`([^`]*)`", r"\1", text)                   # inline code
    text = re.sub(r"\*\*([^*]*)\*\*", r"\1", text)             # bold
    text = re.sub(r"\*([^*]*)\*", r"\1", text)                 # italics
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE) # headers
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.MULTILINE)   # bullets
    text = re.sub(r"^\s*\d+\.\s+", "", text, flags=re.MULTILINE)   # numbered lists
    text = re.sub(r"https?://\S+", "a link", text)             # urls

    # TODO: add substitutions so these are spoken correctly.
    #       Hint: order matters -- do the longest patterns first.
    #         "20%"        -> "20 percent"
    #         "$5"         -> "5 dollars"
    #         "$5 million" -> "5 million dollars"   <- note the word order!
    #         "e.g."       -> "for example"
    #         "i.e."       -> "that is"
    #         "&"          -> "and"
    text = re.sub(r"([\d.]+)\s*%", r"\1 percent", text)
    # magnitude words must be handled BEFORE the bare-number case, or
    # "$5 million" comes out as "5 dollars million".
    text = re.sub(r"\$\s*([\d.,]+)\s+(thousand|million|billion|trillion)",
                  r"\1 \2 dollars", text)
    text = re.sub(r"\$\s*([\d.,]+)", r"\1 dollars", text)
    text = text.replace("e.g.", "for example").replace("i.e.", "that is")
    text = text.replace("&", " and ")

    return re.sub(r"\s+", " ", text).strip()


messy = """## Summary
Here are the **key** points:
- Revenue grew 20% to $5 million
- See `docs.py` or https://example.com/report for details (e.g. the appendix)"""

print("BEFORE:\n" + messy)
print("\nAFTER:\n" + clean_for_speech(messy))

---
# Part 5 - Assembling the loop (60 min)

Three working pieces. Now make them one object.

In [ ]:
import time

class VoiceAssistant:
    """Whisper -> LLM -> TTS, with memory and per-stage timing."""

    def __init__(self, stt_model, system_prompt=SYSTEM_PROMPT, max_turns=6, verbose=True):
        self.stt = stt_model
        self.system = system_prompt
        self.memory = Memory(max_turns=max_turns)
        self.verbose = verbose
        self.timings = []          # one dict per completed turn

    def listen(self, audio_source):
        return robust_transcribe(audio_source, self.stt)

    def think(self, user_text):
        self.memory.add("user", user_text)
        reply, backend = chat(self.memory.messages, self.system)
        reply = clean_for_speech(reply)
        self.memory.add("assistant", reply)
        return reply, backend

    def say(self, text, autoplay=True):
        return speak(text, autoplay=autoplay)

    def turn(self, audio_source, autoplay=True):
        """One full exchange. Returns a dict describing what happened."""
        t = {}

        t0 = time.time()
        user_text = self.listen(audio_source)
        t["stt"] = time.time() - t0

        if not user_text:
            if self.verbose:
                print("(no speech detected -- ignoring)")
            return {"user": "", "reply": "", "timings": t, "skipped": True}

        t0 = time.time()
        reply, backend = self.think(user_text)
        t["llm"] = time.time() - t0

        t0 = time.time()
        self.say(reply, autoplay=autoplay)
        t["tts"] = time.time() - t0

        t["total"] = sum(t.values())
        self.timings.append(t)

        if self.verbose:
            print(f"USER      : {user_text}")
            print(f"ASSISTANT : {reply}")
            print(f"            [{backend}]  stt {t['stt']:.2f}s | "
                  f"llm {t['llm']:.2f}s | tts {t['tts']:.2f}s | total {t['total']:.2f}s")

        return {"user": user_text, "reply": reply, "backend": backend,
                "timings": t, "skipped": False}


assistant = VoiceAssistant(model)
out = assistant.turn(clip_path)

### Where did the time go?

An assistant that takes six seconds to answer feels broken even if every answer is
correct. Rough perceptual targets for turn-taking:

| Total latency | How it feels |
|---|---|
| under 0.5 s | indistinguishable from a person |
| 0.5 - 1.5 s | natural |
| 1.5 - 3 s | noticeably slow but usable |
| over 3 s | the user starts talking again, or repeats themselves |

Profile before you optimise. The stage you *assume* is slow usually is not.

In [ ]:
if assistant.timings:
    t = assistant.timings[-1]
    stages = ["stt", "llm", "tts"]
    values = [t[s] for s in stages]
    labels = ["Whisper\n(speech to text)", "LLM\n(thinking)", "TTS\n(text to speech)"]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(labels, values, color=["#4C72B0", "#DD8452", "#55A868"])
    ax.set_ylabel("seconds")
    ax.set_title(f"Latency breakdown - total {t['total']:.2f}s")
    ax.axhline(1.5, linestyle="--", linewidth=1, color="grey")
    ax.text(2.45, 1.55, "1.5s budget", fontsize=9, color="grey", ha="right")

    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.2f}s\n{v/t['total']*100:.0f}%",
                ha="center", va="bottom", fontsize=10)

    ax.set_ylim(0, max(values) * 1.35)
    plt.tight_layout()
    plt.show()
else:
    print("Run a turn first.")

**Optimisation menu**, roughly in order of payoff:

- **Whisper too slow?** Drop to `tiny.en`, or switch to `faster-whisper` (a CTranslate2
  reimplementation, typically 3-4x quicker for the same weights and less memory).
- **LLM too slow?** Use a smaller local model, cap `max_tokens` harder, and **stream** the
  response so TTS can start on the first sentence instead of waiting for the last.
- **TTS too slow?** Switch from gTTS to offline `pyttsx3` -- this usually removes a full
  second of network round-trip.
- **Everything too slow?** Overlap the stages. Start synthesising sentence one while the
  LLM is still writing sentence two. This is the single biggest perceived-latency win, and
  it is the natural next project after this lab.

In [ ]:
# Multi-turn conversation, driven by synthesised audio so it runs anywhere.
assistant.memory.clear()

script = [
    "Hi there, what can you do?",
    "Tell me one interesting fact about the ocean.",
    "How deep is that?",
]

for line in script:
    src = make_test_clip(line, path=f"{AUDIO_DIR}/turn.wav")
    if src is None:
        print("No TTS engine, so we cannot synthesise test input. Use your mic instead.")
        break
    assistant.turn(src, autoplay=False)
    print()

if assistant.timings:
    avg = sum(t["total"] for t in assistant.timings) / len(assistant.timings)
    print(f"Average turn latency over {len(assistant.timings)} turns: {avg:.2f}s")

Notice the third question -- "How deep is that?" -- only works because of memory. That is
the same pronoun test as Exercise 3.1, now running through the full audio pipeline.

---
# Part 6 - Making it feel real (45 min)

Right now you have to tell the assistant exactly how long to record. Real assistants
figure out on their own when you have stopped talking. That is called **endpointing**, and
a surprisingly effective version needs no machine learning at all.

The idea: chop the incoming audio into short frames (say 30 ms), compute the **RMS energy**
of each frame, and treat frames below a threshold as silence. When you have seen enough
consecutive silent frames *after* some speech, the user is done.

$$\\text{RMS} = \\sqrt{\\frac{1}{N}\\sum_{i=1}^{N} x_i^2}$$

This is the poor cousin of a proper Voice Activity Detector (production systems use
`webrtcvad` or Silero VAD, which handle background noise far better) -- but it is 30 lines,
it is transparent, and it works in a quiet room.

In [ ]:
def rms(frame):
    return float(np.sqrt(np.mean(np.square(frame))))


def frame_energies(audio, sr=SAMPLE_RATE, frame_ms=30):
    n = int(sr * frame_ms / 1000)
    frames = [audio[i:i+n] for i in range(0, len(audio) - n, n)]
    return np.array([rms(f) for f in frames]), frame_ms


energies, frame_ms = frame_energies(audio)
threshold = 0.02          # TODO: tune this for YOUR microphone and room

times = np.arange(len(energies)) * frame_ms / 1000

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(times, energies, linewidth=1.2, label="frame RMS energy")
ax.axhline(threshold, color="crimson", linestyle="--", label=f"threshold = {threshold}")
ax.fill_between(times, 0, energies, where=energies > threshold, alpha=0.3, label="speech")
ax.set_xlabel("time (s)")
ax.set_ylabel("RMS energy")
ax.set_title("Energy-based speech detection")
ax.legend()
ax.margins(x=0)
plt.tight_layout()
plt.show()

speech_frac = (energies > threshold).mean()
print(f"{speech_frac*100:.0f}% of frames classified as speech")
print("If that number looks obviously wrong, adjust `threshold` and re-run.")

In [ ]:
def record_until_silence(sr=SAMPLE_RATE, frame_ms=30, threshold=0.02,
                         silence_ms=800, max_seconds=15,
                         path=f"{AUDIO_DIR}/utterance.wav"):
    """Record until the speaker stops. Returns a path, or None if nobody spoke."""
    import sounddevice as sd
    from scipy.io import wavfile

    frame_len = int(sr * frame_ms / 1000)
    silent_frames_needed = int(silence_ms / frame_ms)
    max_frames = int(max_seconds * 1000 / frame_ms)

    collected, silent_run, started = [], 0, False

    print("Listening... (speak, then pause)")
    with sd.InputStream(samplerate=sr, channels=1, dtype="float32",
                        blocksize=frame_len) as stream:
        for _ in range(max_frames):
            frame, _overflow = stream.read(frame_len)
            frame = frame.flatten()
            collected.append(frame)

            if rms(frame) > threshold:
                started, silent_run = True, 0
            elif started:
                silent_run += 1
                if silent_run >= silent_frames_needed:
                    break

    if not started:
        print("...heard nothing.")
        return None

    audio_out = np.concatenate(collected)
    wavfile.write(path, sr, (audio_out * 32767).astype(np.int16))
    print(f"...captured {len(audio_out)/sr:.1f}s")
    return path


# Uncomment to try it (needs a microphone -- will not work on Colab):
# p = record_until_silence()
# if p: assistant.turn(p)

### The full conversation loop

Put it together and you have a hands-free assistant. Run this as a **`.py` script** rather
than in the notebook -- Jupyter and blocking audio loops do not get along well.

```python
# save as assistant.py, then:  python assistant.py
import whisper
from lab_utils import VoiceAssistant, record_until_silence   # your extracted helpers

model = whisper.load_model("base.en")
bot = VoiceAssistant(model)

print("Say 'goodbye' to exit.\n")
while True:
    path = record_until_silence()
    if path is None:
        continue

    out = bot.turn(path)
    if out["skipped"]:
        continue
    if any(w in out["user"].lower() for w in ("goodbye", "exit", "stop listening")):
        bot.say("Goodbye.")
        break
```

Copy the classes and functions you wrote above into a `lab_utils.py` next to it. Extracting
working notebook code into a module is a good habit in itself -- a notebook is where you
figure something out, not where it lives.

### Exercise 6.1 - add a wake word (20 min)

Always-listening is a privacy problem and a false-trigger problem. Add a wake word: the
assistant transcribes continuously but only *responds* when the transcript starts with a
trigger phrase.

Two approaches, and it is worth understanding why the second one is what real products do:

1. **Transcribe-then-match** (what you can build today): run Whisper on everything, check
   whether the text begins with "computer" or similar. Simple, but you are running a
   240 MB model continuously and shipping every sound in the room through it.
2. **Dedicated wake-word model** (`openWakeWord`, Porcupine): a tiny always-on classifier
   that wakes the heavy pipeline only when triggered. Costs almost nothing to run, and no
   audio leaves the device until you say the word.

Implement approach 1. Then write two sentences on why approach 2 is what Alexa and Siri
actually ship.

In [ ]:
WAKE_WORDS = ("computer", "assistant", "hey bot")

def strip_wake_word(text, wake_words=WAKE_WORDS):
    """Return (was_triggered, command_text_without_the_wake_word)."""
    # TODO: implement. Be forgiving -- Whisper adds punctuation and capitalisation,
    #       so "Computer, what time is it?" must match the wake word "computer".
    low = text.lower().strip().lstrip(" ,.!?")
    for w in wake_words:
        if low.startswith(w):
            # index into `low`, not `text` -- they no longer line up after stripping
            return True, low[len(w):].lstrip(" ,.!?").strip()
    return False, ""


for probe in ["Computer, what time is it?",
              "What time is it?",
              "Hey bot. Tell me a joke."]:
    print(f"{probe!r:<40} -> {strip_wake_word(probe)}")

---
# Part 7 - Stretch challenges (30 min+)

Pick **one** and get it working. These are roughly ordered by difficulty.

**1. Streaming replies.** Stream tokens out of the LLM, buffer until you hit a sentence
boundary, and send each finished sentence to TTS immediately. Measure time-to-first-sound
before and after. This is the biggest perceived-latency win available to you.

**2. A multilingual assistant.** Use Whisper's detected language to answer in the *same*
language the user spoke. Whisper handles the input side for free; you need to pass the
language to both the system prompt and the TTS engine (`gTTS(lang=...)`).

**3. Give it a tool.** Add one real capability -- current time, a timer, a unit converter.
Have the LLM emit a small JSON block when it wants the tool, parse it, run the function,
and feed the result back for a natural-language answer. This is the bridge from "chatbot"
to "agent".

**4. Barge-in.** Let the user interrupt a reply that is still being spoken. Requires
running playback on a separate thread while monitoring mic energy. Fiddly, and the single
thing that most makes an assistant feel real.

**5. Measure your Whisper.** Take 10 clips with known ground-truth transcripts, compute
**Word Error Rate** ($WER = (S + D + I) / N$ -- substitutions plus deletions plus
insertions, over reference words), and compare `tiny.en` / `base.en` / `small.en`. Use the
`jiwer` package. Report accuracy against latency and defend a choice.

**6. Swap in a better voice.** Replace pyttsx3 with a neural TTS such as Piper or Coqui
XTTS. Compare naturalness against latency and installation pain.

---
## Submission checklist

Hand in your completed notebook plus a short README (half a page is plenty).

**Working code**
- [ ] Every TODO cell completed and running
- [ ] `robust_transcribe` returns `''` for silence and correct text for real speech
- [ ] A three-turn conversation where turn 3 depends on memory from turn 1
- [ ] One stretch challenge attempted (say which, and whether it worked)

**Evidence**
- [ ] The model size/latency comparison table from Exercise 2.1, with **your** numbers
- [ ] The latency breakdown chart for your assembled pipeline
- [ ] A saved audio file of your assistant answering something

**Written answers (2-3 sentences each)**
- [ ] Why does Whisper convert audio to a log-Mel spectrogram instead of feeding the raw waveform to the Transformer?
- [ ] Which Whisper model size did you choose for the live loop, and what measurement justifies it?
- [ ] Which pipeline stage dominated your latency, and what is the single change that would help most?
- [ ] Name one specific way your assistant fails, and describe how you would fix it.

---

### Going further

- Whisper paper: *Robust Speech Recognition via Large-Scale Weak Supervision*, Radford et al., 2022
- `openai/whisper` on GitHub -- read `decoding.py` to see the special-token priming in the code
- `faster-whisper` -- the CTranslate2 port you should use for anything real-time
- Silero VAD / `webrtcvad` -- proper voice activity detection, when RMS energy stops being enough